In [ ]:
import os
import re
import uuid
import base64
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime, timedelta, timezone
import logging

logger = logging.getLogger(__name__)

SCOPES = ['https://www.googleapis.com/auth/gmail.modify']

class GmailToolsClass:
    def __init__(self):
        self.service = self._get_gmail_service()
        
    def fetch_unanswered_emails(self, days: int = 0, hours: int = 8, minutes: int = 0, max_results: int = 50):
        """
        Fetch recent unanswered emails.

        Conditions:
        - Email is within the given time window
        - Thread has no existing draft reply
        - Thread is processed only once
        - Self emails are skipped

        Returns:
            List[dict]
        """

        try:
            logger.info(
                f"[gmail] fetching unanswered emails "
                f"(window={days}d {hours}h {minutes}m)"
            )

            # Fetch recent emails
            recent_emails = self.fetch_recent_emails(
                days=days,
                hours=hours,
                minutes=minutes,
                max_results=max_results
            )

            if not recent_emails:
                logger.info("[gmail] no recent emails found")
                return []

            # Fetch draft replies
            drafts = self.fetch_draft_replies(
                days=days,
                hours=hours,
                minutes=minutes
            )

            # Create fast lookup set for threads_with_drafts
            threads_with_drafts = {
                draft.get("threadId")
                for draft in drafts if draft.get("threadId")
            }

            logger.info(
                f"[gmail] threads with drafts: "
                f"{len(threads_with_drafts)}"
            )

            seen_threads = set()
            unanswered_emails = []

            for email in recent_emails:

                try:
                    thread_id = email.get("threadId")
                    email_id = email.get("id")

                    # Skip malformed emails
                    if not thread_id or not email_id:
                        continue

                    # Skip duplicate threads
                    if thread_id in seen_threads:
                        continue

                    # Skip already drafted threads
                    if thread_id in threads_with_drafts:
                        logger.info(
                            f"[gmail] skipped drafted thread "
                            f"(thread_id={thread_id})"
                        )
                        continue

                    seen_threads.add(thread_id)

                    # Fetch full email info
                    email_info = self._get_email_info(email_id)

                    if not email_info:
                        continue

                    self._log_email_read(email_info)

                    # Skip self emails
                    if self._is_email_sent_by_me(email_info):

                        logger.info(
                            f"[gmail] skipped self email "
                            f"(sender={email_info.get('sender')})"
                        )

                        continue

                    unanswered_emails.append(email_info)

                except Exception as email_error:

                    logger.exception(
                        f"[gmail] failed processing email "
                        f"(email_id={email.get('id')}): "
                        f"{email_error}"
                    )

            logger.info(
                f"[gmail] unanswered emails found: "
                f"{len(unanswered_emails)}"
            )

            return unanswered_emails

        except Exception as e:

            logger.exception(
                f"[gmail] failed fetching unanswered emails: "
                f"{e}"
            )

            return []
        
    def fetch_recent_emails(
        self,
        hours: int = 8,
        max_results: int = 50
    ):
        """
        Fetch emails received within the last `hours`.
        """

        try:
            now = datetime.now(timezone.utc)
            start_time = now - timedelta(hours=hours)

            after_timestamp = int(start_time.timestamp())
            before_timestamp = int(now.timestamp())

            query = (
                f"after:{after_timestamp} "
                f"before:{before_timestamp} "
                f"in:inbox"
            )

            response = (
                self.service.users()
                .messages()
                .list(
                    userId="me",
                    q=query,
                    maxResults=max_results
                )
                .execute()
            )

            messages = response.get("messages", [])

            logger.info(
                f"[gmail] fetched {len(messages)} emails "
                f"from last {hours} hours"
            )

            return messages

        except Exception as e:
            logger.exception(
                f"Failed to fetch recent emails: {e}"
            )
            return []
        
    def fetch_draft_replies(self, days: int = 7, hours: int = 0, minutes: int = 0):
        """
        Fetch Gmail draft replies within a given time window.

        Returns:
            List[dict]
        """

        try:
            logger.info(
                f"[gmail] fetching drafts "
                f"(window={days}d {hours}h {minutes}m)"
            )

            # Calculate time window
            now = datetime.now(timezone.utc)

            start_time = now - timedelta(
                days=days,
                hours=hours,
                minutes=minutes
            )

            drafts_response = (
                self.service.users()
                .drafts()
                .list(userId="me")
                .execute()
            )

            draft_list = drafts_response.get("drafts", [])

            logger.info(
                f"[gmail] total drafts fetched: "
                f"{len(draft_list)}"
            )

            filtered_drafts = []

            for draft in draft_list:

                try:
                    message = draft.get("message", {})

                    internal_date = message.get("internalDate")

                    if not internal_date:
                        continue

                    draft_time = datetime.fromtimestamp(
                        int(internal_date) / 1000,
                        tz=timezone.utc
                    )

                    if draft_time < start_time:
                        continue

                    filtered_drafts.append({
                        "draft_id": draft.get("id"),
                        "threadId": message.get("threadId"),
                        "id": message.get("id"),
                        "created_at": draft_time.isoformat()
                    })

                except Exception as draft_error:
                    logger.exception(
                        f"[gmail] failed processing draft: "
                        f"{draft_error}"
                    )

            logger.info(
                f"[gmail] filtered drafts count: "
                f"{len(filtered_drafts)}"
            )

            return filtered_drafts

        except Exception as error:
            logger.exception(
                f"[gmail] failed fetching drafts: {error}"
            )
            return []

    def create_draft_reply(self, initial_email, reply_text):
        try:
            # Create the reply message
            message = self._create_reply_message(initial_email, reply_text)

            # Create draft with thread information
            draft = self.service.users().drafts().create(
                userId="me", body={"message": message}
            ).execute()

            return draft
        except Exception as error:
            print(f"An error occurred while creating draft: {error}")
            return None

    def send_reply(self, initial_email, reply_text):
        try:
            # Create the reply message
            message = self._create_reply_message(initial_email, reply_text, send=True)

            # Send the message with thread ID
            sent_message = self.service.users().messages().send(
                userId="me", body=message
            ).execute()
            
            return sent_message

        except Exception as error:
            print(f"An error occurred while sending reply: {error}")
            return None
        
    def _create_reply_message(self, email, reply_text, send=False):
        # Create message with proper headers
        message = self._create_html_email_message(
            recipient=email.sender,
            subject=email.subject,
            reply_text=reply_text
        )

        # Set threading headers
        if email.messageId:
            message["In-Reply-To"] = email.messageId
            # Combine existing references with the original message ID
            message["References"] = f"{email.references} {email.messageId}".strip()
            
            if send:
                # Generate a new Message-ID for this reply
                message["Message-ID"] = f"<{uuid.uuid4()}@gmail.com>"
                
        # Construct email body
        body = {
            "raw": base64.urlsafe_b64encode(message.as_bytes()).decode(),
            "threadId": email.threadId
        }

        return body

        
    def _get_gmail_service(self):
        creds_path = os.environ.get('CREDENTIALS_PATH', 'credentials.json')
        token_path = os.environ.get('TOKEN_PATH', 'token.json')
        
        creds = None
        if os.path.exists(token_path):
            creds = Credentials.from_authorized_user_file(token_path, SCOPES)
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(creds_path, SCOPES)
                creds = flow.run_local_server(port=0)
            with open(token_path, 'w') as token:
                token.write(creds.to_json())
        return build('gmail', 'v1', credentials=creds) 
    
    def _is_email_sent_by_me(self, email_info):
        return os.environ['MY_EMAIL'] in email_info['sender']

    def _get_email_info(self, msg_id):
        message = self.service.users().messages().get(
            userId="me", id=msg_id, format="full"
        ).execute()

        payload = message.get('payload', {})
        headers = {header["name"].lower(): header["value"] for header in payload.get("headers", [])}

        return {
            "id": msg_id,
            "threadId": message.get("threadId"),
            "messageId": headers.get("message-id"),
            "references": headers.get("references", ""),
            "sender": headers.get("from", "Unknown"),
            "subject": headers.get("subject", "No Subject"),
            "body": self._get_email_body(payload),
        }
    
    def _get_email_body(self, payload):
        """
        Extract the email body, prioritizing text/plain over text/html.
        Handles multipart messages, avoids duplicating content, and strips HTML if necessary.
        """
        def decode_data(data):
            """Decode base64-encoded data."""
            return base64.urlsafe_b64decode(data).decode('utf-8').strip() if data else ""

        def extract_body(parts):
            """Recursively extract text content from parts."""
            for part in parts:
                mime_type = part.get('mimeType', '')
                data = part['body'].get('data', '')
                if mime_type == 'text/plain':
                    return decode_data(data)
                if mime_type == 'text/html':
                    html_content = decode_data(data)
                    return self._extract_main_content_from_html(html_content)
                if 'parts' in part:
                    result = extract_body(part['parts'])
                    if result:
                        return result
            return ""

        # Process single or multipart payload
        if 'parts' in payload:
            body = extract_body(payload['parts'])
        else:
            data = payload['body'].get('data', '')
            body = decode_data(data)
            if payload.get('mimeType') == 'text/html':
                body = self._extract_main_content_from_html(body)

        return self._clean_body_text(body)

    def _extract_main_content_from_html(self, html_content):
        """
        Extract main visible content from HTML.
        """
        soup = BeautifulSoup(html_content, 'html.parser')
        for tag in soup(['script', 'style', 'head', 'meta', 'title']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)

    def _clean_body_text(self, text):
        """
        Clean up the email body text by removing extra spaces and newlines.
        """
        return re.sub(r'\s+', ' ', text.replace('\r', '').replace('\n', '')).strip()
    
    def _create_html_email_message(self, recipient, subject, reply_text):
        """
        Creates a simple HTML email message with proper formatting and plaintext fallback.
        """
        message = MIMEMultipart("alternative")
        message["to"] = recipient
        message["subject"] = f"Re: {subject}" if not subject.startswith("Re: ") else subject

        # Simplified HTML Template
        html_text = reply_text.replace("\n", "<br>").replace("\\n", "<br>")
        html_content = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <meta charset="utf-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0">
        </head>
        <body>{html_text}</body>
        </html>
        """

        html_part = MIMEText(html_content, "html")

        # message.attach(text_part)
        message.attach(html_part)

        return message
    

gmail = GmailToolsClass()


In [6]:
msg = gmail.fetch_recent_emails(5)

In [7]:
print(msg)

[{'id': '19e686993ba78ce9', 'threadId': '19e686993ba78ce9'}, {'id': '19e6867277c43be8', 'threadId': '19e6867277c43be8'}]
